# Overcooked-AI — Pipeline Colab · Etapas 3-4 (PPO + finetune + multi-layout)

Config-driven vía `CONFIG`:
- **Esc.1** vs greedy: `esc1.yaml` · **Esc.2** vs greedy+sticky: `esc2.yaml`
- **Esc.3** vs greedy+sticky+eps: `esc3.yaml` · **Esc.4** cocinar solo (GENERALISTA multi-layout): `esc4.yaml`

**Regla `numpy<2`:** tras instalar, **REINICIAR EL RUNTIME** una vez. Árbitro: el **harness oficial**.


## Celda 1 — Clonar repo + instalar


In [ ]:
import os
REPO_URL = 'https://github.com/Snah-s/deep_project.git'
REPO_DIR = '/content/deep_project'
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!git pull   # trae checkpoints de etapas previas (init_from del finetune)
!bash scripts/setup_colab.sh

## ⚠️ REINICIA EL RUNTIME AHORA
`Entorno de ejecución` → `Reiniciar sesión`. Sigue desde la **Celda 2**.


## Celda 2 — Verificar entorno (GATE 0)


In [ ]:
%cd /content/deep_project
!python -m pytest tests/test_env_smoke.py -q

## Celda 3 — Montar Drive (persistencia)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE_OUT = '/content/drive/MyDrive/overcooked_ckpts'
os.makedirs(DRIVE_OUT, exist_ok=True)
print('Checkpoints ->', DRIVE_OUT)

## Celda 4 — Configuración

`LAYOUT=None` usa el/los layout(s) del config (para Esc.4 es el POOL multi-layout).
Pon un nombre en `LAYOUT` solo para forzar un especialista de un layout.


In [ ]:
CONFIG = 'training/configs/esc1.yaml'   # esc1 | esc2 | esc3 | esc4
LAYOUT = None                           # None = layout(s) del config; o 'nombre' para especialista
TOTAL_TIMESTEPS = None                  # None = del yaml; o un int
N_ENVS = 8
VEC = 'subproc'                         # 'dummy' si SubprocVecEnv falla en Colab
INIT_FROM = None                        # None = del yaml; o ruta a checkpoint

## Celda 5 — Entrenamiento (PPO / finetune / multi-layout)


In [ ]:
%cd /content/deep_project
extra = ''
if LAYOUT:          extra += f' --layout {LAYOUT}'
if TOTAL_TIMESTEPS: extra += f' --total-timesteps {TOTAL_TIMESTEPS}'
if INIT_FROM:       extra += f" --init-from '{INIT_FROM}'"
cmd = (f"python -m training.train_ppo --config {CONFIG}"
       f" --n-envs {N_ENVS} --vec {VEC} --device auto --output-dir '{DRIVE_OUT}'{extra}")
print(cmd)
!{cmd}

## Celda 6 — Evaluar con el harness oficial

Evalúa cada layout del pool vs el compañero del escenario Y vs greedy limpio (regresión).


In [ ]:
%cd /content/deep_project
import sys, yaml; sys.path.insert(0, 'overcooked')
from evaluation.harness import evaluate, _summary
from envs.partners import make_partner

cfg = yaml.safe_load(open(CONFIG))
pool = cfg['layout'] if isinstance(cfg['layout'], list) else [cfg['layout']]
if LAYOUT: pool = [LAYOUT]
tag = pool[0] if len(pool) == 1 else 'multi'
run_name = f"{cfg['experiment_name']}_{tag}"
best_path = f'{DRIVE_OUT}/{run_name}/best_model'
eval_partner = cfg['eval']['partner_spec']
agent = lambda: make_partner({'type': 'checkpoint', 'path': best_path})
for lay in pool:
    r_sc = evaluate(agent, lay, eval_partner, seeds=[67, 68, 69])
    r_gr = evaluate(agent, lay, {'type': 'greedy'}, seeds=[67, 68, 69])
    print(f'[{lay}] vs escenario:', _summary(r_sc))
    print(f'[{lay}] vs greedy   :', _summary(r_gr))

## Umbrales de los GATES
- **GATE 3** (esc1): `soups_mean ≥ 1` vs greedy y score ≥ baseline greedy+greedy.
- **GATE 4** (esc2/esc3): `soups_mean ≥ 2` vs greedy+sticky.
- **GATE 4** (esc4): `soups_mean ≥ 1` vs random_motion, en cada layout del pool.

Perillas si un gate no pasa (en orden): `shaping.anneal_fraction`→0.8, luego duplicar pasos. Anotar en EXPERIMENTS.md.


## Celda 7 (opcional) — Curva de score


In [ ]:
import json, matplotlib.pyplot as plt
hist = json.load(open(f'{DRIVE_OUT}/{run_name}/eval_history.json'))
xs = [h['timesteps'] for h in hist]; ys = [h['score_mean'] for h in hist]
plt.plot(xs, ys, marker='o'); plt.xlabel('timesteps'); plt.ylabel('harness score_mean (prom. pool)')
plt.title(run_name); plt.grid(True); plt.show()